# 08 RAG Index Build\n
Create embeddings and build vector index.

In [ ]:
import json
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful")

# Initialize directories
corpus_path = Path("../data/corpus")
index_path = corpus_path / "faiss_index"
index_path.mkdir(parents=True, exist_ok=True)

print(f"Using corpus directory: {corpus_path}")

In [ ]:
print("\n" + "="*60)
print("Building FAISS Index")
print("="*60)

# Create FAISS index
embedding_dim = embeddings.shape[1]
print(f"\nCreating FAISS IndexFlatL2 with dimension {embedding_dim}")

index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings.astype('float32'))

print(f"✓ Index built with {index.ntotal} vectors")

# Save index
index_file = index_path / "financial_corpus.index"
faiss.write_index(index, str(index_file))
print(f"✓ Saved FAISS index to {index_file}")

# Save embedding model reference
with open(corpus_path / "embedding_model.txt", 'w') as f:
    f.write("all-MiniLM-L6-v2")

print(f"\n✓ RAG Index Ready for Querying!")
print(f"  - Corpus size: {len(corpus_docs)} documents")
print(f"  - Embedding dim: {embedding_dim}")
print(f"  - Index type: L2 (Euclidean distance)")

## 4. Build FAISS Index

In [ ]:
print("\n" + "="*60)
print("Generating Embeddings")
print("="*60)

# Load embedding model
print("\nLoading embedding model: all-MiniLM-L6-v2")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings (batch process for efficiency)
print(f"Embedding {len(corpus_docs)} documents...")
batch_size = 32
embeddings = []

for batch_start in range(0, len(corpus_docs), batch_size):
    batch_end = min(batch_start + batch_size, len(corpus_docs))
    batch_docs = corpus_docs[batch_start:batch_end]
    
    batch_embeddings = embedding_model.encode(batch_docs, convert_to_numpy=True)
    embeddings.extend(batch_embeddings)
    
    print(f"  Processed {batch_end}/{len(corpus_docs)} documents")

embeddings = np.array(embeddings)
print(f"\n✓ Generated embeddings: {embeddings.shape}")
print(f"  Dimension: {embeddings.shape[1]}")
print(f"  Documents: {embeddings.shape[0]}")

## 3. Generate Embeddings

In [ ]:
print("\n" + "="*60)
print("Building Financial Corpus")
print("="*60)

# Load training data as corpus
train_path = Path("../data/processed/train.jsonl")
corpus_docs = []
corpus_metadata = []

print(f"\nLoading training data as corpus...")
with open(train_path, 'r') as f:
    for idx, line in enumerate(f):
        sample = json.loads(line)
        # Use context as main document
        doc_text = sample['context']
        if doc_text.strip():  # Only add non-empty docs
            corpus_docs.append(doc_text)
            corpus_metadata.append({
                'doc_id': idx,
                'question': sample['instruction'],
                'answer': sample['output']
            })

print(f"✓ Loaded {len(corpus_docs)} financial documents")

# Save metadata for later reference
metadata_file = corpus_path / "metadata.jsonl"
with open(metadata_file, 'w') as f:
    for meta in corpus_metadata:
        f.write(json.dumps(meta) + '\n')
print(f"✓ Saved metadata to {metadata_file}")

## 2. Create Financial Corpus from Training Data